In [ ]:
# %pip install requests bs4

In [1]:
import requests
import re
import json
import os
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone



def get_category_pages(category_url):

    headers = {
        "User-Agent": "YourBot/1.0 (https://example.com/contact)"
    }
    
    res = requests.get(category_url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")

    # Extract category name from URL
    url_parse = urlparse(category_url)
    path = url_parse.path
    category_name = path.split(":")[-1]

    base = url_parse.scheme + "://" + url_parse.netloc # https://simple.wikipedia.org
    pages = []

    for li in soup.select("#mw-pages li a"):
        href = li.get("href")
        title = li.get_text(strip=True)
        if "Template:" in title:
            continue
        if href and href.startswith("/wiki/"):
            pages.append({
                "title": title,
                "url": base + href
            })

    return {
        "categories": category_name,
        "category_urls": category_url,
        "pages": pages
    }

# category_url = "https://simple.wikipedia.org/wiki/Category:Statistics"
# get_category_pages(category_url=category_url)



def clean_spaces(text):
    """Utility to clean extra spaces and newlines."""
    return " ".join(text.split())

def scrape_simple_wiki(url):
    headers = {
        "User-Agent": "ReverseMentorBot/0.1 (https://yourdomain.com/contact)"
    }
    stop_sections = {"references", 
                     "other websites", 
                     "related pages", 
                     "further reading",
                     "external links"}

    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")

    # Check for redirect pages
    redirect_div = soup.find("div", class_="redirectMsg")
    if redirect_div and redirect_div.find("a"):
        redirect_url = "https://simple.wikipedia.org" + redirect_div.find("a")["href"]
        return {
            "url": url,
            "title": None,
            "sections": [],
            "redirect": redirect_url
        }

    # Main content container
    content = soup.find("div", class_="mw-parser-output")
    if not content:
        return {
            "url": url,
            "title": None,
            "sections": [],
            "error": "Main content not found"
        }

    # Remove citation superscripts like [1]
    for sup in content.find_all("sup"):
        sup.decompose()

    # Article title
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else None

    article_data = {
        "url": url,
        "title": title,
        "sections": [],
        "categories": [],
        "category_urls": [],
    }

    # --- Extract intro paragraphs ---
    intro_section = {"heading": "Introduction", "paragraphs": []}
    
    # All sections including intro (section[0] usually the intro)
    sections = content.find_all("section", recursive=False)
    for section in sections:
        # Get section heading if exists
        h2 = section.find("h2")
        heading = h2.get_text(strip=True) if h2 else None

        # Skip sections like references, other websites
        if heading and heading.lower() in stop_sections:
            continue

        # Set current section
        current_section = {"heading": heading if heading else "Introduction", "paragraphs": []}

        # Extract all paragraphs recursively
        for child in section.descendants:
            if child.name == "p":
                text = clean_spaces(child.get_text(" ", strip=True))
                if text:
                    current_section["paragraphs"].append(text)

            # Extract bullet/numbered lists
            elif child.name in ("ul", "ol"):
                for index, li in enumerate(child.find_all("li", recursive=False), start=1):
                    text = clean_spaces(li.get_text(" ", strip=True))
                    if not text:
                        continue
                    if child.name == "ul":
                        current_section["paragraphs"].append(f"- {text}")
                    else:
                        current_section["paragraphs"].append(f"{index}) {text}")

        # Only add section if it has content
        if current_section["paragraphs"]:
            article_data["sections"].append(current_section)

    # --- Categories ---
    for cat in soup.select("#mw-normal-catlinks ul li a"):
        article_data["categories"].append(cat.get_text(strip=True))
        article_data["category_urls"].append("https://simple.wikipedia.org" + cat.get("href"))

    return article_data



#---
# test_page_url = 'https://simple.wikipedia.org/wiki/Software'
# scrape_simple_wiki(test_page_url)
#---



def scrape_category_parallel(category_url, max_workers=10):
    """
    Scrape all pages in a Simple Wikipedia category in parallel
    and store each page in the JSON cache with category info.
    """
    result = []

    category_data = get_category_pages(category_url)
    cat_title = category_data["categories"]
    cat_url = category_data["category_urls"]

    page_urls = [p["url"] for p in category_data["pages"]]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        scraped_pages = list(executor.map(scrape_simple_wiki, page_urls))

    for scraped in scraped_pages:
        entry = {
            "title": scraped["title"],
            "url": scraped["url"],
            "sections": scraped["sections"],
            "categories": [cat_title],
            "category_urls": [cat_url]
        }
        result.append(entry)

    return result

# test_category = "https://simple.wikipedia.org/wiki/Category:Statistics"
# result = scrape_category_parallel(test_category)
# result

def scrape_all_pages(urls, max_workers=10):
    
	result = []

	with ThreadPoolExecutor(max_workers=max_workers) as executor:
		scraped_pages = list(executor.map(scrape_simple_wiki, urls))
	
	for scraped in scraped_pages:
		entry = {
            "title": scraped["title"],
            "url": scraped["url"],
            "sections": scraped["sections"],
			"categories": [],
            "category_urls": []
        }
		result.append(entry)
	
	return result

# test_pages = ["https://simple.wikipedia.org/wiki/Software", "https://simple.wikipedia.org/wiki/Machine_learning"]
# result = scrape_all_pages(test_pages)
# result
def scrape_all_categories(category_urls, cat_workers=5, page_workers=10):

    results = []

    def scrape_single_category(url):
        return scrape_category_parallel(url, max_workers=page_workers)

    with ThreadPoolExecutor(max_workers=cat_workers) as executor:
        category_results = list(executor.map(scrape_single_category, category_urls))

    # Flatten the list of lists
    for r in category_results:
        results.extend(r)

    return results

# test_categories = ["https://simple.wikipedia.org/wiki/Category:Statistics", "https://simple.wikipedia.org/wiki/Category:Artificial_intelligence"]
# result = scrape_all_categories(test_categories)
# result
CACHE_FILE = "simple_wiki_raw_data.json"


def load_cache():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_cache(cache):
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2, ensure_ascii=False)


def store_scrape(article_dict):
    """
    Store scraped article in JSON cache.
    Preserves and merges categories and category_urls from older entries.
    Updates timestamps intelligently.
    """
    cache = load_cache()
    url = article_dict["url"]

    # Find old entry if present
    old = next((item for item in cache if item["url"] == url), None)
    cache = [item for item in cache if item["url"] != url]

    now = datetime.now(timezone.utc).isoformat()
    article_dict["last_scraped"] = now

    if old:
        # Keep original first-scrape
        article_dict["first_scraped"] = old.get("first_scraped", now)

        # === Merge categories ===
        old_cats = set(old.get("categories", []))
        new_cats = set(article_dict.get("categories", []))
        article_dict["categories"] = list(old_cats | new_cats)

        # === Merge category URLs ===
        old_cat_urls = set(old.get("category_urls", []))
        new_cat_urls = set(article_dict.get("category_urls", []))
        article_dict["category_urls"] = list(old_cat_urls | new_cat_urls)

    else:
        # First time scraping
        article_dict["first_scraped"] = now

        # Normalize fields to lists if missing
        article_dict["categories"] = article_dict.get("categories", [])
        article_dict["category_urls"] = article_dict.get("category_urls", [])

    # Save back
    cache.append(article_dict)
    save_cache(cache)

# helpers
def is_page_url(x: str) -> bool:
    return isinstance(x, str) \
        and x.startswith("https://simple.wikipedia.org/wiki/") \
        and "Category:" not in x

def is_category_url(x: str) -> bool:
    return isinstance(x, str) \
        and x.startswith("https://simple.wikipedia.org/wiki/Category:")
    
def is_page_url_list(x) -> bool:
    return isinstance(x, list) and len(x) > 0 and all(is_page_url(i) for i in x)

def is_category_url_list(x) -> bool:
    return isinstance(x, list) and len(x) > 0 and all(is_category_url(i) for i in x)


# handler functions
def handle_page_url(url):
    return [scrape_simple_wiki(url)]

def handle_category_url(category_url, max_workers=10):
    return scrape_category_parallel(category_url, max_workers)

def handle_page_url_list(urls, max_workers=10):
    return scrape_all_pages(urls, max_workers)

def handle_category_url_list(category_urls, cat_workers=5, page_workers=10):
    return scrape_all_categories(category_urls, cat_workers, page_workers)

# handlers
handlers = [
    (is_page_url, handle_page_url),
    (is_category_url, handle_category_url),
    (is_page_url_list, handle_page_url_list),
    (is_category_url_list, handle_category_url_list),
]

# Main scraping function
def scrape(x, store=True):
    for predicate, handler in handlers:
        if predicate(x):
            result_pages = handler(x)
            if store:
                  for page in result_pages:
                        store_scrape(page)
            return result_pages
                
    raise ValueError(f"Unknown input type: {x!r}")

# res_test = scrape(test_categories, store=True)
# len(res_test), res_test

In [2]:
# Manually gathered simple Wiki categories

category_urls = ["https://simple.wikipedia.org/wiki/Category:Statistics",
                 "https://simple.wikipedia.org/wiki/Category:Artificial_intelligence",
                 "https://simple.wikipedia.org/wiki/Category:Probability_theory",
                 "https://simple.wikipedia.org/wiki/Category:Probability_distributions",
                 "https://simple.wikipedia.org/wiki/Category:Graph_theory",
                 "https://simple.wikipedia.org/wiki/Category:Theoretical_computer_science",
                 "https://simple.wikipedia.org/wiki/Category:Algorithms",
                 "https://simple.wikipedia.org/wiki/Category:Data_compression",
                 "https://simple.wikipedia.org/wiki/Category:Greedy_algorithms",
                 "https://simple.wikipedia.org/wiki/Category:Numerical_analysis",
                 "https://simple.wikipedia.org/wiki/Category:Randomised_algorithms",
                 "https://simple.wikipedia.org/wiki/Category:Searching_and_sorting_algorithms",
                 "https://simple.wikipedia.org/wiki/Category:Cryptography",
                 "https://simple.wikipedia.org/wiki/Category:Algebra",
                 "https://simple.wikipedia.org/wiki/Category:Geometry",
                 "https://simple.wikipedia.org/wiki/Category:Logic",
                 "https://simple.wikipedia.org/wiki/Category:Number_theory",
                 "https://simple.wikipedia.org/wiki/Category:Mathematical_analysis",
                 "https://simple.wikipedia.org/wiki/Category:Calculus",
                 "https://simple.wikipedia.org/wiki/Category:Game_theory",
                 ]


scrape("https://simple.wikipedia.org/wiki/Software")
scrape("https://simple.wikipedia.org/wiki/Generative_model")
scrape("https://simple.wikipedia.org/wiki/Generative_model")
scrape("https://simple.wikipedia.org/wiki/Kernel_(computer_science)","https://simple.wikipedia.org/wiki/Kernel_(computer_science)")
scrape("https://simple.wikipedia.org/wiki/Category:Artificial_intelligence")
scrape('https://simple.wikipedia.org/wiki/Nim')
scrape(["https://simple.wikipedia.org/wiki/XAI",
        'https://simple.wikipedia.org/wiki/Nim',
        "https://simple.wikipedia.org/wiki/Virtual_assistant",
        "https://simple.wikipedia.org/wiki/Virtual_assistant"])

result = scrape(category_urls)
len(result), result

(799,
 [{'title': 'Regression toward the mean',
   'url': 'https://simple.wikipedia.org/wiki/Regression_toward_the_mean',
   'sections': [{'heading': 'Introduction',
     'paragraphs': ['Regression toward the mean simply means that, following an extreme random event, the next random event is likely to be less extreme. Regression toward the mean was first described by Francis Galton . He found that offspring of tall parents tended to be shorter. Also, offspring of shorter parents tended to be taller. Galton stated that processes that did not follow regression towards the mean would quickly go out of control .']},
    {'heading': 'History',
     'paragraphs': ["In 1886, Galton published a paper called Regression towards mediocrity in hereditary stature . In the paper, he observed that extreme characteristics (e.g., height) in parents are not passed on completely to their offspring. Rather, the characteristics in the offspring regress towards a mediocre point. Today, this point is called 

In [3]:
no_title_items = [item for item in result if item.get("title") is None]
no_title_items


[{'title': None,
  'url': 'https://simple.wikipedia.org/wiki/IPsec',
  'sections': [],
  'categories': ['Cryptography'],
  'category_urls': ['https://simple.wikipedia.org/wiki/Category:Cryptography'],
  'last_scraped': '2026-01-28T07:47:09.579094+00:00',
  'first_scraped': '2026-01-28T07:47:09.579094+00:00'},
 {'title': None,
  'url': 'https://simple.wikipedia.org/wiki/Kerckhoffs%27s_principle',
  'sections': [],
  'categories': ['Cryptography'],
  'category_urls': ['https://simple.wikipedia.org/wiki/Category:Cryptography'],
  'last_scraped': '2026-01-28T07:47:09.627578+00:00',
  'first_scraped': '2026-01-28T07:47:09.627578+00:00'},
 {'title': None,
  'url': 'https://simple.wikipedia.org/wiki/Key_(cryptography)',
  'sections': [],
  'categories': ['Cryptography'],
  'category_urls': ['https://simple.wikipedia.org/wiki/Category:Cryptography'],
  'last_scraped': '2026-01-28T07:47:09.650555+00:00',
  'first_scraped': '2026-01-28T07:47:09.650555+00:00'},
 {'title': None,
  'url': 'https://

In [4]:
# don't understand why these have no section
no_sections_items = [item for item in result if not item.get("sections")]
no_sections_items


[{'title': 'Seven Bridges of Königsberg',
  'url': 'https://simple.wikipedia.org/wiki/Seven_Bridges_of_K%C3%B6nigsberg',
  'sections': [],
  'categories': ['Graph_theory'],
  'category_urls': ['https://simple.wikipedia.org/wiki/Category:Graph_theory'],
  'last_scraped': '2026-01-28T07:47:06.440530+00:00',
  'first_scraped': '2026-01-28T07:47:06.440530+00:00'},
 {'title': 'Bletchley Park',
  'url': 'https://simple.wikipedia.org/wiki/Bletchley_Park',
  'sections': [],
  'categories': ['Cryptography'],
  'category_urls': ['https://simple.wikipedia.org/wiki/Category:Cryptography'],
  'last_scraped': '2026-01-28T07:47:08.710778+00:00',
  'first_scraped': '2026-01-28T07:47:08.710778+00:00'},
 {'title': None,
  'url': 'https://simple.wikipedia.org/wiki/IPsec',
  'sections': [],
  'categories': ['Cryptography'],
  'category_urls': ['https://simple.wikipedia.org/wiki/Category:Cryptography'],
  'last_scraped': '2026-01-28T07:47:09.579094+00:00',
  'first_scraped': '2026-01-28T07:47:09.579094+00: